In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import polars as pl
import unicodedata
import re
from functools import partial
import random
import time
import math
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

c:\Users\Benjamin\dev\nlp\slangify\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [4]:
df = pl.read_csv("urbandict-word-defs.csv", has_header=True, truncate_ragged_lines=True, infer_schema_length=0)

df_clean = df.filter(
    pl.col("definition").is_not_null() & 
    pl.col("word").is_not_null()
)

df_clean = df_clean.with_columns([
    pl.col("up_votes").cast(pl.Int64, strict=False),
    pl.col("down_votes").cast(pl.Int64, strict=False)
])

df_clean = df_clean.drop_nulls(subset=["up_votes", "down_votes"])


In [4]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normaliseString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s) # Add space before punctuation
    s = re.sub(r"[^a-zA-Z!?<>]+", r" ", s) # Keep only letters and !?
    return s.strip()

In [ ]:
class CharacterTokenizer:
    def __init__(self, texts=None):
        self.special_tokens = ["<pad>", "<unk>", "<sos>", "<eos>"]

        self.char2index = {tok: i for i, tok in enumerate(self.special_tokens)}
        self.index2char = {i: tok for i, tok in enumerate(self.special_tokens)}
        self.n_chars = len(self.special_tokens)
        if texts is not None:
            self.build_vocab(texts)

    def build_vocab(self, texts):
        for text in texts:
            normalised = normaliseString(text)
            for word in normalised.split():
                self.add_word(word)            

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1       

    def add_char(self, char):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

In [ ]:
class WordTokenizer:
    def __init__(self, texts=None):
        self.special_tokens = ["<pad>", "<unk>", "<sos>", "<eos>"]

        self.word2index = {tok: i for i, tok in enumerate(self.special_tokens)}
        self.index2word = {i: tok for i, tok in enumerate(self.special_tokens)}
        self.n_words = len(self.special_tokens)
        if texts is not None:
            self.build_vocab(texts)

    def build_vocab(self, texts):
        for text in texts:
            normalised = normaliseString(text)
            for word in normalised.split():
                self.add_word(word)
            

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

    def encode(self, text):
        normalised = normaliseString(text)
        tokens = normalised.split()
        ids = [self.word2index.get(tok, self.word2index["<unk>"]) for tok in tokens]
        return [self.word2index["<sos>"]] + ids + [self.word2index["<eos>"]]
    
    def decode(self, ids):
        words = [self.index2word.get(id, "<unk>") for id in ids]
        return " ".join([w for w in words if w not in self.special_tokens])

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        return [self.encode(text) for text in texts]
    
    @property
    def pad_token_id(self):
        return self.word2index["<pad>"] 
       
    @property
    def sos_token_id(self):
        return self.word2index["<sos>"]
    
    @property
    def eos_token_id(self):
        return self.word2index["<eos>"]
    
    @property
    def vocab_size(self):
        return self.n_words

In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, inputs, outputs, tokenizer, batch_size=1):

        self.input_ids = tokenizer(inputs)
        self.output_ids = tokenizer(outputs)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {"input_ids": torch.tensor(self.input_ids[idx],dtype=torch.long), "output_ids": torch.tensor(self.output_ids[idx],dtype=torch.long)}

In [5]:
class FlanDataset(Dataset):
    def __init__(self, inputs, outputs, tokenizer, batch_size=1):
        self.input_ids = []
        self.input_attention_masks = []
        self.target_ids = []
        self.target_attention_masks = []

        for i in range(0, len(inputs), batch_size):
            batch_input = tokenizer(inputs[i:i + batch_size], return_tensors="pt", truncation=True, padding=True)
            batch_target = tokenizer(outputs[i:i + batch_size], return_tensors="pt", truncation=True, padding=True)

            self.input_ids.extend(batch_input["input_ids"])
            self.input_attention_masks.extend(batch_input["attention_mask"])
            self.target_ids.extend(batch_target["input_ids"])
            self.target_attention_masks.extend(batch_target["attention_mask"])


    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
                    "input_ids": self.input_ids[idx],
                    "input_attention_masks": self.input_attention_masks[idx],
                    "target_ids": self.target_ids[idx],
                    "target_attention_masks": self.target_attention_masks[idx]
                }

In [61]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

definitions  = df_clean["definition"].to_list()[:10000]
words  = df_clean["word"].to_list()[:10000]
batch_size = 64
dataset  = FlanDataset(definitions, words, tokenizer, batch_size=batch_size)

In [148]:
first_entry = dataset[0]

In [149]:
print(tokenizer.decode(first_entry["input_ids"]))
print(tokenizer.decode(first_entry["target_ids"]))

Undesirable; less-than optimum.</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
Janky</s><pad><pad><pad><pad><pad><pad><pad>


In [80]:
def collate_fn(batch, pad_token_id):
    batch_input_ids = [item["input_ids"] for item in batch]
    batch_output_ids = [item["output_ids"] for item in batch]

    padded_inputs = nn.utils.rnn.pad_sequence(
        batch_input_ids, 
        padding_value=pad_token_id,
        batch_first=True
    ).to(device)

    padded_outputs = nn.utils.rnn.pad_sequence(
        batch_output_ids, 
        padding_value=pad_token_id,
        batch_first=True
    ).to(device)
    return {"input_ids": padded_inputs, "output_ids": padded_outputs}

In [81]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.gru = nn.GRU(embedding_size, hidden_size, batch_first=True)

    def forward(self, input):
        embedded = self.embedding(input)
        outputs, hidden = self.gru(embedded)

        # outputs: [batch_size, seq_length, hidden_size]
        # hidden: [1, batch_size, hidden_size]
        return outputs, hidden

In [82]:
class Attention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, encoder_outputs, hidden):

        hidden = hidden.permute(1,0,2) # [batch_size, 1, hidden_size]
        attention_scores = torch.bmm(hidden, encoder_outputs.transpose(1,2)) # [batch_size, 1, seq_length]
        attention_weights = F.softmax(attention_scores,dim=2) # [batch_size, 1, seq_length]
        attention_vector = torch.bmm(attention_weights, encoder_outputs) # [batch_size, 1, hidden_size]

        return attention_vector, attention_weights

In [83]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, hidden_size)
        self.W_v = nn.Linear(hidden_size, hidden_size)

        self.scale = hidden_size ** 0.5

    def forward(self, encoder_outputs, hidden):

        hidden = hidden.permute(1,0,2) # [batch_size, 1, hidden_size]

        Q = self.W_q(hidden)
        K = self.W_k(encoder_outputs)
        V = self.W_v(encoder_outputs)

        attention_scores = torch.bmm(Q, K.transpose(1,2)) / self.scale # [batch_size, 1, seq_length]
        attention_weights = F.softmax(attention_scores,dim=2) # [batch_size, 1, seq_length]
        attention_vector = torch.bmm(attention_weights, V) # [batch_size, 1, hidden_size]

        return attention_vector, attention_weights

In [84]:
class DecoderRNN(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size, attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.attention = attention
        self.gru = nn.GRU(embedding_size + hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(self, encoder_outputs, hidden, input):
        embedded = self.embedding(input)
        attention_vector, _ = self.attention(encoder_outputs, hidden)
        rnn_input = torch.concat([embedded, attention_vector], dim=2)
        output, hidden = self.gru(rnn_input)
        output = self.out(output)

        return output, hidden


In [85]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, input, target_tensor=None):
        encoder_outputs, encoder_hidden = self.encoder(input)
        batch_size = input.size(0)
        
        if target_tensor is not None:
            seq_length = target_tensor.size(1)
        else:
            seq_length = 5

        decoder_hidden = encoder_hidden
        decoder_input = torch.empty((batch_size, 1), dtype=torch.long, device=device).fill_(tokenizer.sos_token_id)
        decoder_outputs = []

        for i in range(seq_length):
            decoder_output, decoder_hidden = self.decoder(encoder_outputs, decoder_hidden, decoder_input)
            decoder_outputs.append(decoder_output)

            _, topi = torch.topk(decoder_output, 1)
            decoder_input = topi.squeeze(-1).detach()


        decoder_outputs = torch.concat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)


        return decoder_outputs, decoder_hidden

In [14]:
def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [7]:
def train_epoch(dataloader, model, optimizer, criterion):
    total_loss = 0
    for batch in dataloader:
        input, input_attentions, labels = batch["input_ids"].to(device), batch["input_attention_masks"].to(device), batch["target_ids"].to(device)
        optimizer.zero_grad()

        outputs = model(input, attention_mask=input_attentions, labels=labels)
        # loss = criterion(
        #     outputs.view(-1, outputs.size(-1)),
        #     labels.view(-1)
        # )
        loss = outputs[0]
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

In [8]:
def train(dataloader, model, n_epochs, lr=0.001, print_every=100):
    start = time.time()
    print_loss_total = 0

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.NLLLoss(ignore_index=tokenizer.pad_token_id)

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(dataloader, model, optimizer, criterion)
        print_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))


In [69]:
# definitions  = df_clean["definition"].to_list()[:100000]
# words  = df_clean["word"].to_list()[:100000]

# tokenizer  = WordTokenizer(definitions  + words )
# dataset  = SimpleDataset(definitions , words , tokenizer )

# print("Vocab size: ", tokenizer.vocab_size)

In [52]:
# embedding_size = 768
# hidden_size = 512

# encoder = EncoderRNN(tokenizer.vocab_size, embedding_size, hidden_size)

# attention = Attention()
# decoder = DecoderRNN(tokenizer.vocab_size, embedding_size, hidden_size, attention)

# model = Seq2Seq(encoder, decoder).to(device)

model_original = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(device)

In [ ]:
# batch_size = 512
n_epochs = 50
# dataloader  = DataLoader(dataset , collate_fn=partial(collate_fn, pad_token_id=tokenizer .pad_token_id), batch_size=batch_size, shuffle=True)
dataloader = DataLoader(dataset, batch_size=batch_size)
train(dataloader, model, n_epochs, print_every=2)

In [54]:
def evaluate(model, text):
    with torch.no_grad():
        input = tokenizer(text, return_tensors="pt").to(device)
        outputs = model.generate(**input)
        return tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # _, topi = torch.topk(outputs, 1)
        # decoded_ids = topi.squeeze().tolist()

        # decoded_text = tokenizer.decode(decoded_ids)
        # return decoded_text

def evaluate_original(model, text):
    with torch.no_grad():
        input = tokenizer(text, return_tensors="pt")
        outputs = model.generate(**input)
        return tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # _, topi = torch.topk(outputs, 1)
        # decoded_ids = topi.squeeze().tolist()

        # decoded_text = tokenizer.decode(decoded_ids)
        # return decoded_text


In [60]:
text = "someone who eats pussy"

result = evaluate(model, text)
print(result)

print(evaluate_original(model_original, text))

['splork']
['is a person who eats pussy.']
